# LSTM

In [11]:
# ================================
# LSTM: run per-store (no scaling leakage), log1p target, report errors in dollars
# Trains one model per store and aggregates MAE/RMSE
# ================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

# Settings - tune as needed
seq_len = 10
epochs = 30  # change if you want longer training
batch_size = 32

# Helper to compute Thanksgiving (4th Thursday of November)
def thanksgiving_date(year):
    d = pd.Timestamp(year=year, month=11, day=1)
    first_thursday = 1 + ((3 - d.weekday()) % 7)
    return pd.Timestamp(year=year, month=11, day=first_thursday + 21).date()

# Load data
df = pd.read_csv("Walmart.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")
df = df.sort_values(["Store", "Date"]).reset_index(drop=True)

# Compute holiday features (major: Thanksgiving, Christmas). Use Holiday_Flag if available to mark other holidays.
years = df['Date'].dt.year.unique()
thanksgiving_dates = {int(y): thanksgiving_date(int(y)) for y in years}

def is_major(d):
    if (d.month == 12 and d.day == 25):
        return 1
    ty = int(d.year)
    if thanksgiving_dates.get(ty) == d.date():
        return 1
    return 0

df['is_major_holiday'] = df['Date'].apply(is_major)
# if dataset has Holiday_Flag (1 indicates holiday week), use it to mark minor holidays (excluding major)
if 'Holiday_Flag' in df.columns:
    df['is_minor_holiday'] = ((df['Holiday_Flag'] == 1) & (df['is_major_holiday'] == 0)).astype(int)
else:
    df['is_minor_holiday'] = 0

# Monthly cyclical features
df['Month'] = df['Date'].dt.month
df['month_sin'] = np.sin(2 * np.pi * (df['Month'] - 1) / 12)
df['month_cos'] = np.cos(2 * np.pi * (df['Month'] - 1) / 12)

# Features to use
features = ['is_major_holiday', 'is_minor_holiday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'month_sin', 'month_cos']
target = 'Weekly_Sales'

stores = sorted(df['Store'].unique())
results = []

for store_id in stores:
    print(f"\nProcessing store {store_id}")
    store_df = df[df['Store'] == store_id].reset_index(drop=True)
    n = len(store_df)
    if n <= seq_len + 5:
        print(f"Skipping store {store_id}: not enough rows ({n})")
        continue

    raw_X = store_df[features].values
    raw_y = store_df[[target]].values.flatten()

    # sequences counts and splits per-store
    num_seq = n - seq_len
    train_seq = int(0.7 * num_seq)
    val_seq = int(0.15 * num_seq)
    test_seq = num_seq - train_seq - val_seq
    fit_end = seq_len + train_seq

    # target log transform with safety shift
    min_y = raw_y.min()
    shift = float(-min_y + 1.0) if min_y < 0 else 0.0
    if shift > 0:
        print(f"Store {store_id}: applying shift={shift} to target before log1p")
    raw_y_shifted = raw_y + shift
    raw_y_log = np.log1p(raw_y_shifted)

    # Fit scalers on training portion only (no leakage)
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()
    scaler_X.fit(raw_X[:fit_end])
    scaler_y.fit(raw_y_log[:fit_end].reshape(-1, 1))

    X_scaled = scaler_X.transform(raw_X)
    y_scaled = scaler_y.transform(raw_y_log.reshape(-1, 1))

    # Create sequences
    def create_sequences(X, y, seq_len=10):
        Xs, ys = [], []
        for i in range(seq_len, len(X)):
            Xs.append(X[i-seq_len:i])
            ys.append(y[i])
        return np.array(Xs), np.array(ys)

    X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_len)

    X_train = X_seq[:train_seq]
    y_train = y_seq[:train_seq]
    X_val = X_seq[train_seq:train_seq+val_seq]
    y_val = y_seq[train_seq:train_seq+val_seq]
    X_test = X_seq[train_seq+val_seq:]
    y_test = y_seq[train_seq+val_seq:]

    print(f"Store {store_id}: total sequences={num_seq}, train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")

    if len(X_train) == 0 or len(X_test) == 0:
        print(f"Skipping store {store_id}: empty train/test after split")
        continue

    n_features = X_train.shape[2]

    # Build model
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(128, return_sequences=True, input_shape=(seq_len, n_features)))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(LSTM(64))
    model.add(BatchNormalization())
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

    # Callbacks and checkpoint
    ckpt = f"best_model_store_{store_id}.weights.h5"
    callbacks = [
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
        ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True, save_weights_only=True, verbose=1)
    ]

    # Train
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, validation_data=(X_val, y_val), callbacks=callbacks, verbose=1)

    # Load best weights if checkpoint exists
    if os.path.exists(ckpt):
        model.load_weights(ckpt)
        print(f"Loaded best weights from {ckpt}")

    # Predict and invert transforms back to dollars
    y_pred_scaled = model.predict(X_test)
    y_pred_log = scaler_y.inverse_transform(y_pred_scaled)
    y_pred_orig = np.expm1(y_pred_log).flatten() - shift

    y_test_log = scaler_y.inverse_transform(y_test)
    y_test_orig = np.expm1(y_test_log).flatten() - shift

    mae = mean_absolute_error(y_test_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
    print(f"Store {store_id} - MAE: ${mae:.2f}, RMSE: ${rmse:.2f}")

    # Save per-store predictions and metrics
    out_df = pd.DataFrame({"y_test_dollars": y_test_orig, "y_pred_dollars": y_pred_orig})
    out_csv = f"store_{store_id}_predictions.csv"
    out_df.to_csv(out_csv, index=False)
    print(f"Saved predictions to: {out_csv}")

    final_model_path = f"store_{store_id}_model.h5"
    try:
        model.save(final_model_path)
        print(f"Saved full model to {final_model_path}")
    except Exception as e:
        print("Could not save full model (h5). You can save weights separately:", e)

    results.append({'store': store_id, 'mae': mae, 'rmse': rmse, 'n_test': len(y_test_orig)})

# Aggregate results
if len(results) == 0:
    raise RuntimeError('No stores processed')

res_df = pd.DataFrame(results)
simple_mae = res_df['mae'].mean()
simple_rmse = res_df['rmse'].mean()
weighted_mae = (res_df['mae'] * res_df['n_test']).sum() / res_df['n_test'].sum()
weighted_rmse = (res_df['rmse'] * res_df['n_test']).sum() / res_df['n_test'].sum()

print('\nSummary:')
print(f'Store count processed: {len(res_df)}')
print(f'Simple average MAE=${simple_mae:.2f}, RMSE=${simple_rmse:.2f}')
print(f'Weighted MAE=${weighted_mae:.2f}, RMSE=${weighted_rmse:.2f}')

res_df.to_csv('per_store_metrics.csv', index=False)
print('Saved per_store_metrics.csv')



Processing store 1
Store 1: total sequences=133, train=93, val=19, test=21




Epoch 1/30


c:\Users\arpit\anaconda3\envs\sales_forecasting\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.8372
Epoch 1: val_loss improved from None to 0.09454, saving model to best_model_store_1.weights.h5

Epoch 1: val_loss improved from None to 0.09454, saving model to best_model_store_1.weights.h5
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 316ms/step - loss: 0.8857 - val_loss: 0.0945 - learning_rate: 0.0010
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 316ms/step - loss: 0.8857 - val_loss: 0.0945 - learning_rate: 0.0010
Epoch 2/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.6052
Epoch 2: val_loss did not improve from 0.09454
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.6350 - val_loss: 0.1107 - learning_rate: 0.0010
Epoch 3/30

Epoch 2: val_loss did not improve from 0.09454
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.6350 - val_loss: 0.1107 - learning_rate: 0.0010
Epoch 3/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 0.6261
Epoch 3: val_loss did not improve from 0.09454
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.4591 - val_loss: 0.1266 - lear

Store 1 - MAE: $200583.58, RMSE: $223917.52
Saved predictions to: store_1_predictions.csv
Saved full model to store_1_model.h5

Processing store 2
Store 2: total sequences=133, train=93, val=19, test=21
Epoch 1/30
Epoch 1/30


...

Epoch 1/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step - loss: 0.9053
Epoch 1: val_loss improved from None to 0.35737, saving model to best_model_store_44.weights.h5

Epoch 1: val_loss improved from None to 0.35737, saving model to best_model_store_44.weights.h5
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 202ms/step - loss: 0.8370 - val_loss: 0.3574 - learning_rate: 0.0010
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 202ms/step - loss: 0.8370 - val_loss: 0.3574 - learning_rate: 0.0010
Epoch 2/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - loss: 0.7374
Epoch 2: val_loss did not improve from 0.35737
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.7114 - val_loss: 0.4296 - learning_rate: 0.0010
Epoch 3/30

Epoch 2: val_loss did not improve from 0.35737
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 0.7114 - val_loss: 0.4296 - learning_rate: 0.0010
Epoch 3/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.6144
Epoch 3: val_loss did not improve from 0.35737
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.6792 - val_loss: 0

Store 44 - MAE: $88835.42, RMSE: $89346.25
Saved predictions to: store_44_predictions.csv
Saved full model to store_44_model.h5

Processing store 45
Store 45: total sequences=133, train=93, val=19, test=21
Epoch 1/30
Epoch 1/30


c:\Users\arpit\anaconda3\envs\sales_forecasting\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/3 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step - loss: 2.4274
Epoch 1: val_loss improved from None to 0.01760, saving model to best_model_store_45.weights.h5

Epoch 1: val_loss improved from None to 0.01760, saving model to best_model_store_45.weights.h5
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 310ms/step - loss: 1.8391 - val_loss: 0.0176 - learning_rate: 0.0010
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 310ms/step - loss: 1.8391 - val_loss: 0.0176 - learning_rate: 0.0010
Epoch 2/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 0.8608
Epoch 2: val_loss did not improve from 0.01760
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.6037 - val_loss: 0.0309 - learning_rate: 0.0010
Epoch 3/30

Epoch 2: val_loss did not improve from 0.01760
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.6037 - val_loss: 0.0309 - learning_rate: 0.0010
Epoch 3/30
1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.5398
Epoch 3: val_loss did not improve from 0.01760
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.4523 - val_loss: 0.0496 - lear

Store 45 - MAE: $40335.65, RMSE: $53806.52
Saved predictions to: store_45_predictions.csv
Saved full model to store_45_model.h5

Summary:
Store count processed: 45
Simple average MAE=$147256.95, RMSE=$158781.37
Weighted MAE=$147256.95, RMSE=$158781.37
Saved per_store_metrics.csv
